# Local NISAR GUNW to pairwise dSWE

This is SnowIn's canonical real-product workflow. It requires a local NISAR GUNW and a prepared NISAR-modified Copernicus DEM; it does not search for products, download inputs, or acquire a DEM.

Install the optional integrations and notebook tools with `python -m pip install -e ".[nisar,geometry,dask]" --group notebooks`.

In [ ]:
import os
from pathlib import Path

from IPython.display import display

from snowin import compute_leinss_dswe
from snowin.io import add_gunw_incidence, open_gunw

gunw_path = Path(os.environ["SNOWIN_GUNW"]).expanduser().resolve()
dem_path = Path(os.environ["SNOWIN_NISAR_DEM"]).expanduser().resolve()
if not gunw_path.is_file():
    raise FileNotFoundError(gunw_path)
if not dem_path.is_file():
    raise FileNotFoundError(dem_path)
print("GUNW:", gunw_path)
print("prepared DEM:", dem_path)

## 1. Open the phase-normalized product

`open_gunw()` normalizes the source phase and reads authoritative wavelength metadata. It does **not** compute incidence. The returned ordinary xarray Dataset remains in `phase_normalized_product` state. Available correction fields are exposed with provenance, including native radar-grid screens, and are never applied automatically.

In [ ]:
pair = open_gunw(gunw_path, chunks="auto")
assert pair.attrs["snowin_data_state"] == "phase_normalized_product"
print(
    {
        "data_state": pair.attrs["snowin_data_state"],
        "phase_definition": pair.phase.attrs["phase_difference_definition"],
        "source_phase_definition": pair.attrs["source_phase_difference_definition"],
        "reference_time": pair.attrs["reference_time"],
        "secondary_time": pair.attrs["secondary_time"],
        "temporal_edge": pair.attrs["temporal_edge"],
        "wavelength_m": pair.attrs["wavelength_m"],
        "wavelength_source": pair.attrs["wavelength_source"],
    }
)
display(pair)

## 2. Add local incidence and retrieve pairwise dSWE

Local incidence is an explicit geometry operation using the prepared DEM and GUNW line-of-sight vectors. The call updates this resource-owning Dataset in place. Invalid or back-facing geometry remains missing and is masked before retrieval. The result is pairwise SWE change between the radar epochs, not absolute SWE.

In [ ]:
try:
    pair = add_gunw_incidence(
        pair,
        gunw_path,
        dem=dem_path,
        dem_source="nisar_cop30",
        require_vertical_datum_match=True,
    )
    assert pair.attrs["snowin_data_state"] == "retrieval_ready_pair"
    assert pair.phase.attrs["phase_difference_definition"] == "secondary_minus_reference"
    assert pair.incidence_angle.attrs["incidence_angle_reference"] == "local"

    print(
        {
            "data_state": pair.attrs["snowin_data_state"],
            "incidence_reference": pair.incidence_angle.attrs[
                "incidence_angle_reference"
            ],
            "geometry_supported_fraction": float(pair.geometry_valid.mean()),
            "vertical_datum_status": pair.attrs["vertical_datum_status"],
        }
    )
    display(pair[["phase", "incidence_angle", "geometry_valid"]])

    dswe = compute_leinss_dswe(
        pair.phase.where(pair.geometry_valid),
        pair.incidence_angle.where(pair.geometry_valid),
        wavelength_m=float(pair.attrs["wavelength_m"]),
    )
    display(dswe)
    print("pairwise dSWE units:", dswe.attrs["units"])
finally:
    pair.close()